# 01 Data Analysis - NSL-KDD Intrusion Detection

This notebook performs exploratory data analysis for the NSL-KDD dataset.

Data source:
- `data/raw/archive/nsl-kdd/KDDTrain+.txt`
- `data/raw/archive/nsl-kdd/KDDTest+.txt`

Goal:
- Build a clean EDA pipeline for binary intrusion detection (`normal` vs `attack`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")

# NSL-KDD column names: 41 features + attack + difficulty_level
NSL_KDD_COLUMNS = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "attack",
    "difficulty_level",
]

assert len(NSL_KDD_COLUMNS) == 43, "Expected 43 NSL-KDD columns."

In [ ]:
candidate_dirs = [
    Path("../data/raw/archive/nsl-kdd"),
    Path("../data/raw/archive(2)/nsl-kdd"),
    Path("../data/raw/archive(2)"),
]

data_dir = None
for candidate in candidate_dirs:
    if (candidate / "KDDTrain+.txt").exists() and (candidate / "KDDTest+.txt").exists():
        data_dir = candidate
        break

if data_dir is None:
    raise FileNotFoundError(
        "Could not locate NSL-KDD files. Expected KDDTrain+.txt and KDDTest+.txt in known data/raw paths."
    )

train_path = data_dir / "KDDTrain+.txt"
test_path = data_dir / "KDDTest+.txt"

train_df = pd.read_csv(train_path, header=None, names=NSL_KDD_COLUMNS)
test_df = pd.read_csv(test_path, header=None, names=NSL_KDD_COLUMNS)

print("Using data directory:", data_dir)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain dtypes:")
print(train_df.dtypes)

print("\nTrain first 5 rows:")
display(train_df.head())

print("\nTest first 5 rows:")
display(test_df.head())

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\archive\\nsl-kdd\\KDDTrain+.txt'

In [ ]:
def add_binary_target(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    normalized_attack = out["attack"].astype(str).str.strip().str.rstrip(".")
    out["attack_binary"] = np.where(normalized_attack.eq("normal"), 0, 1)
    out = out.drop(columns=["difficulty_level"])
    return out

train_df = add_binary_target(train_df)
test_df = add_binary_target(test_df)

print("Updated train columns:", train_df.shape[1])
print("Updated test columns:", test_df.shape[1])
print("attack_binary value counts (train):")
print(train_df["attack_binary"].value_counts().sort_index())
print("attack_binary value counts (test):")
print(test_df["attack_binary"].value_counts().sort_index())

Shape: (5000, 23)

New default rate: 0.1460
Shock_intensity correlation with default: 0.2027
structural_risk_score correlation with default: 0.2972
Proportion with shock_intensity > 0.6: 0.4148
Max absolute feature correlation with default: 0.3865
Perfect-correlation check (>=0.99 abs): False
Selected calibration -> scale: 2.1765, intercept: -5.3082


In [ ]:
train_size = len(train_df)
test_size = len(test_df)
num_features = train_df.shape[1] - 1  # exclude attack_binary target

train_attack_count = int((train_df["attack_binary"] == 1).sum())
train_normal_count = int((train_df["attack_binary"] == 0).sum())
attack_rate = float(train_df["attack_binary"].mean())

print("Training dataset size:", train_size)
print("Testing dataset size:", test_size)
print("Number of features:", num_features)
print("Number of attack samples (train):", train_attack_count)
print("Number of normal samples (train):", train_normal_count)
print(f"Attack rate (train): {attack_rate:.4f}")

Top 10 feature correlations with default:

loan_to_income           0.386530
monthly_income          -0.302302
structural_risk_score    0.297241
temporary_stress         0.273819
shock_intensity          0.202748
loan_amount              0.180392
income_drop_pct          0.178105
income_drop_gt_30        0.160346
adequate_savings        -0.056254
relief_received         -0.040383
Name: default, dtype: float64


In [ ]:
categorical_features = ["protocol_type", "service", "flag"]
excluded_cols = {"attack", "attack_binary"}
feature_columns = [c for c in train_df.columns if c not in excluded_cols]
numeric_features = [c for c in feature_columns if c not in categorical_features]

print("Number of categorical features:", len(categorical_features))
print("Number of numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)

In [ ]:
# 1) Attack vs Normal distribution
plt.figure(figsize=(6, 4))
attack_counts = train_df["attack_binary"].value_counts().sort_index()
sns.barplot(x=attack_counts.index, y=attack_counts.values, hue=attack_counts.index, palette="Set2", legend=False)
plt.xticks([0, 1], ["Normal (0)", "Attack (1)"])
plt.title("Attack vs Normal Distribution (Train)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# 2) Protocol distribution
plt.figure(figsize=(7, 4))
protocol_counts = train_df["protocol_type"].value_counts()
sns.barplot(x=protocol_counts.index, y=protocol_counts.values, hue=protocol_counts.index, palette="crest", legend=False)
plt.title("Protocol Type Distribution (Train)")
plt.xlabel("Protocol")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# 3) Service distribution (top 10)
plt.figure(figsize=(10, 5))
service_counts = train_df["service"].value_counts().head(10)
sns.barplot(x=service_counts.index, y=service_counts.values, hue=service_counts.index, palette="viridis", legend=False)
plt.title("Top 10 Services (Train)")
plt.xlabel("Service")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 4) Flag distribution
plt.figure(figsize=(8, 4))
flag_counts = train_df["flag"].value_counts()
sns.barplot(x=flag_counts.index, y=flag_counts.values, hue=flag_counts.index, palette="mako", legend=False)
plt.title("Flag Distribution (Train)")
plt.xlabel("Flag")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# 5) Histogram of src_bytes
plt.figure(figsize=(8, 4))
sns.histplot(train_df["src_bytes"], bins=50, kde=True, color="steelblue")
plt.title("Histogram of src_bytes (Train)")
plt.xlabel("src_bytes")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# 6) Histogram of dst_bytes
plt.figure(figsize=(8, 4))
sns.histplot(train_df["dst_bytes"], bins=50, kde=True, color="darkorange")
plt.title("Histogram of dst_bytes (Train)")
plt.xlabel("dst_bytes")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# 7) Correlation heatmap for numeric features
plt.figure(figsize=(16, 12))
heatmap_df = train_df[numeric_features + ["attack_binary"]].corr()
sns.heatmap(heatmap_df, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Numeric Features + Target)")
plt.tight_layout()
plt.show()

In [ ]:
corr_with_target = (
    train_df[numeric_features + ["attack_binary"]]
    .corr()["attack_binary"]
    .drop("attack_binary")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print("Top 10 numeric features correlated with attack_binary:\n")
print(corr_with_target.head(10))

In [ ]:
combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

output_path = Path("../data/processed/nsl_kdd_processed.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
combined_df.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")

In [ ]:
final_rows, final_cols = combined_df.shape
final_attack_ratio = float(combined_df["attack_binary"].mean())

print("Final NSL-KDD Processed Dataset Summary")
print("=" * 45)
print("Dataset rows:", final_rows)
print("Dataset columns:", final_cols)
print(f"Attack ratio: {final_attack_ratio:.4f}")
print("Categorical feature list:", categorical_features)
print("Numeric feature list:", numeric_features)